In [1]:
import pandas as pd

# ARROW summaries (posetail-next/arrow_baseline: arrow_cached.sh, arrow_cached_po.sh), paper-protocol metrics
ARROW_ROOT = '/groups/karashchuk/home/karashchukl/results/paper-final-v2/arrow-outputs'

# units as in format_tables_v2.ipynb: AJ / delta_avg / OA in %, MTE in cm for panoptic and dex_ycb,
# x13 for kubric, raw for point odyssey
MTE_SCALE = {'cmupanoptic_3dgs': 100, 'dex_ycb': 100, 'kubric-multiview': 13,
             'point-odyssey-human': 1, 'po-animal-sub256-motion': 1}


def load_arrow(folder):
    """Headline metrics [AJ, delta_avg, OA, MTE] per dataset, in table units."""
    df = pd.read_csv(f'{ARROW_ROOT}/{folder}/combined_summary_metrics.csv').set_index('dataset')
    return {ds: [row['arrow_avg_jaccard'] * 100, row['arrow_delta_x_avg'] * 100,
                 row['arrow_occlusion_acc'] * 100, row['arrow_mte'] * MTE_SCALE[ds]]
            for ds, row in df.iterrows()}


def fmt(values):
    return ' & '.join(f'{v:.1f}' for v in values)

In [2]:
# tab:main_results: Panoptic, DexYCB, Kubric (4 canonical cameras)
main = load_arrow('paper-canonical-cached')
nums = [v for ds in ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview'] for v in main[ds]]
print(f'ARROW~\\cite{{fradlin2026arrow}} & See~\\cite{{fradlin2026arrow}} & \\xmark& {fmt(nums)} \\\\')

ARROW~\cite{fradlin2026arrow} & See~\cite{fradlin2026arrow} & \xmark& 69.1 & 77.6 & 92.0 & 8.1 & 49.6 & 55.6 & 90.9 & 20.1 & 30.0 & 35.2 & 88.7 & 9.2 \\


In [3]:
# tab:po_results: 4 views (human, animal), then all 9 views (human, animal)
po4, po9 = load_arrow('paper-po-4cams'), load_arrow('paper-po-9cams')
nums = [v for res in (po4, po9) for ds in ['point-odyssey-human', 'po-animal-sub256-motion'] for v in res[ds]]
print(f'ARROW~\\cite{{fradlin2026arrow}}\n  & {fmt(nums)} \\\\')

ARROW~\cite{fradlin2026arrow}
  & 28.4 & 34.1 & 91.2 & 5.2 & 38.6 & 45.3 & 93.5 & 3.2 & 38.9 & 43.7 & 95.6 & 4.0 & 38.6 & 45.7 & 96.5 & 2.6 \\


### figures with ARROW
Saved to `figures/arrow/`.

In [ ]:
import os, sys, glob
import cv2
import matplotlib
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import gridspec

matplotlib.rcParams.update({'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix'})

# inputs of figure_ncams.ipynb / figure_survival.ipynb / figure_qualitative_comparison.ipynb, plus the ARROW results
# tracktail results with visibility threshold 0.1 (posetail-next TRACKTAIL_VIS_THRESH); the other methods' columns
# are copied from /home/ruppk2@hhmi.org/tracktail-paper/data/v2
PAPER_DATA = '/groups/karashchuk/home/karashchukl/results/paper-final-v2/posetail-outputs-vis0.1/paper_data'
POSETAIL_NEXT = '/groups/karashchuk/home/karashchukl/projects/posetail/posetail-next'
ARROW_RENDERS = '/groups/karashchuk/home/karashchukl/renders/arrow-renders/paper-canonicalcams-cached-100'
FIG_OUT = 'figures/arrow'
os.makedirs(FIG_OUT, exist_ok=True)

palette = sns.color_palette('colorblind', n_colors=12)
ARROW_COLOR = palette[4]

In [ ]:
# n_cams_figure.pdf: DexYCB AJ vs number of views (figure_ncams.ipynb) with ARROW added
data = pd.read_csv(f'{PAPER_DATA}/dexycb_ncam_results.csv')
arrow_ncam = pd.read_csv(f'{ARROW_ROOT}/ncam-ablation-dexycb/ncam_ablation_summary.csv')
data = data.merge(pd.DataFrame({'n_views': arrow_ncam['n_views'],
                                'ARROW': arrow_ncam['arrow_avg_jaccard'] * 100}), on='n_views', how='left')

method_dict = {'CoTracker3 Baseline': 'CoTracker3 Baseline',
               'MVTracker Baseline': 'MVTracker (DUST3R)',
               'ARROW': 'ARROW',
               'Tracktail': 'Tracktail'}
method_order = ['Tracktail', 'CoTracker3 Baseline', 'MVTracker Baseline', 'ARROW']
colors = [palette[2], palette[9], palette[3], ARROW_COLOR]

sns.set_style('ticks', {'font.family': 'STIXGeneral'})
df = data.melt(id_vars='n_views', var_name='method', value_name='avg_jaccard')
df['method'] = pd.Categorical(df['method'], categories=method_order, ordered=True)
df = df.sort_values('method')

fs = 17
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
sns.lineplot(data=df, x='n_views', y='avg_jaccard', hue='method', marker='o', markersize=7, palette=colors, ax=ax)
ax.set_xlabel('Number of Views', fontsize=fs)
ax.set_ylabel('Avg Jaccard (AJ)', fontsize=fs)
ax.set_title('Performance on DexYCB', fontsize=fs)
ax.set_xticks(sorted(df['n_views'].unique()))
ax.tick_params(axis='both', labelsize=fs)
ax.get_legend().remove()
ax.grid(True, linestyle=':', color='k', linewidth=0.5, alpha=0.8)
ax.set_ylim([0, 90])   # Leave room for the legend below the data

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [method_dict[m] for m in labels], loc='lower right',
          framealpha=0.7, facecolor='white', edgecolor='lightgray', fontsize=fs - 3)
plt.tight_layout()
sns.despine()
fig.savefig(os.path.join(FIG_OUT, 'n_cams_figure.pdf'), dpi=800, bbox_inches='tight')

In [ ]:
# po_survival_comparison_1.2.pdf: survival on MV Point Odyssey, 4 cams (figure_survival.ipynb) with ARROW added.
# The ARROW curves are computed with posetail-next/scripts/compute_survival.py's functions, as for the other methods.
sys.path.insert(0, f'{POSETAIL_NEXT}/scripts')
import compute_survival as cs

thresh = 1.2
data = pd.read_csv(f'{PAPER_DATA}/survival_curves.csv')
arrow_rows = []
for dataset, (folder, prefix) in cs.DATASETS.items():
    cfg = dict(model='arrow-4cams', dataset=dataset, pattern=f'{ARROW_ROOT}/paper-po-4cams/{folder}/{prefix}*_trial.npz')
    arrow_rows += [dict(r, threshold=thresh) for r in cs.process_config(cfg, thresh)]
data = pd.concat([data, pd.DataFrame(arrow_rows)], ignore_index=True)

label_dict = {'posetail-4cams': 'Tracktail', 'cotracker-4cams': 'CoTracker Baseline',
              'mvtracker-4cams-duster': 'MVTracker', 'arrow-4cams': 'ARROW'}
color_dict = dict(zip(label_dict, [palette[2], palette[9], palette[3], ARROW_COLOR]))

sns.set_style('ticks', {'font.family': 'STIXGeneral'})
fs = 20
fig, axs = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
threshold_df = data[(data['threshold'] == thresh) & data['model'].isin(label_dict)]
for i, (ax, dataset) in enumerate(zip(axs, ['human', 'animal'])):
    dataset_df = threshold_df[threshold_df['dataset'] == dataset]
    sns.lineplot(data=dataset_df, x='frame', y='survival_mean', hue='model', hue_order=list(label_dict),
                 linewidth=2, palette=color_dict, ax=ax)
    for model in label_dict:
        sub = dataset_df[dataset_df['model'] == model].sort_values('frame')
        ax.fill_between(sub['frame'], sub['survival_mean'] - sub['survival_std'],
                        sub['survival_mean'] + sub['survival_std'], color=color_dict[model], alpha=0.15, linewidth=0)
    ax.set_xlabel('Frame', fontsize=fs)
    ax.set_ylabel('Survival Rate' if i == 0 else '', fontsize=fs)
    ax.set_title(f'Multiview Point Odyssey {dataset.capitalize()}', fontsize=fs)
    ax.tick_params(axis='both', labelsize=fs)
    ax.get_legend().remove()
    ax.grid(True, linestyle=':', color='k', linewidth=0.5, alpha=0.8)
    ax.set_xlim([0, 500])
    ax.set_ylim([0, 1])

handles, labels = axs[-1].get_legend_handles_labels()
fig.legend(handles, [label_dict[l] for l in labels], loc='upper center', bbox_to_anchor=(0.365, 0.92),
           framealpha=0.7, facecolor='white', edgecolor='lightgray', fontsize=fs - 2)
plt.tight_layout()
plt.subplots_adjust(wspace=0.15)
sns.despine()
fig.savefig(os.path.join(FIG_OUT, f'po_survival_comparison_{thresh}.pdf'), dpi=800, bbox_inches='tight')

In [ ]:
# qualitative_comparison.pdf (figure_qualitative_comparison.ipynb, as of 4467be9) with an ARROW column.
# ARROW predictions: posetail-next/arrow_baseline/arrow_100.sh.
dex_trial = '20200918-subject-06__20200918_114117'
cmu_trial = 'tennis'
# renders of scripts/render_qualitative_gtvis.sh (posetail-next): every method shows/hides a point by the
# ground-truth visibility, so all columns show the same points at every frame
G = '/groups/karashchuk/home/karashchukl/renders/gtvis-renders'
video_prefixes = {
    ds: {'CoTracker3 Baseline': f'{G}/cotracker/{d}/{trial}',
         'MVTracker':           f'{G}/mvtracker/{d}/{trial}',
         'ARROW':               f'{G}/arrow/{d}/{trial}',
         'Tracktail':           f'{G}/tracktail/{d}/{trial}',
         'Ground Truth':        f'{G}/gt/{d}/{trial}'}   # last, next to Tracktail
    for ds, d, trial in [('dex-ycb', 'dex_ycb', dex_trial), ('cmupanoptic_3dgs', 'cmupanoptic_3dgs', cmu_trial)]
}
methods = list(video_prefixes['dex-ycb'])
specs = [{'dataset': 'dex-ycb', 'views': ['view_00'], 'frames': [11, 23]},
         {'dataset': 'cmupanoptic_3dgs', 'views': ['14'], 'frames': [89, 149]}]


def read_frame(path, frame_num, target_ar=16 / 9):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ok, img = cap.read()
    cap.release()
    if not ok:
        raise FileNotFoundError(f'cannot read frame {frame_num} of {path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]   # center-crop to 16:9 so all rows share the same aspect ratio
    if w / h < target_ar:
        new_h = int(round(w / target_ar)); y0 = (h - new_h) // 2; img = img[y0:y0 + new_h]
    elif w / h > target_ar:
        new_w = int(round(h * target_ar)); x0 = (w - new_w) // 2; img = img[:, x0:x0 + new_w]
    return img


fs = 20
sns.set_style('ticks', {'font.family': 'STIXGeneral'})
n_cols = len(methods)
fig = plt.figure(figsize=(3.5 * n_cols, 10))
gs = gridspec.GridSpec(4, n_cols, wspace=0.0, hspace=0.0, left=0.05, right=0.95)
axes = [[fig.add_subplot(gs[row, col]) for col in range(n_cols)] for row in range(4)]
for row_idx, row_axes in enumerate(axes):
    spec = specs[row_idx // 2]
    frame_num = spec['frames'][row_idx % 2]
    for col_idx, ax in enumerate(row_axes):
        method = methods[col_idx]
        ax.imshow(read_frame(os.path.join(video_prefixes[spec['dataset']][method], f"{spec['views'][0]}.mp4"), frame_num))
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if row_idx == 0:
            ax.set_title(method, fontsize=fs)
        if col_idx == 0:
            ax.set_ylabel(f't = {frame_num + 1}', fontsize=fs - 1)

for dataset_idx, dataset_name in enumerate(['Dex YCB', 'Panoptic Studio']):
    top, bottom = axes[2 * dataset_idx][0].get_position(), axes[2 * dataset_idx + 1][0].get_position()
    fig.text(0.015, (top.y0 + bottom.y1) / 2, dataset_name, ha='center', va='center', rotation=90,
             fontsize=fs + 2, transform=fig.transFigure)
fig.savefig(os.path.join(FIG_OUT, 'qualitative_comparison.pdf'), dpi=800, bbox_inches='tight')